# Instructions, ModelSettings, and Structured Output

Our Leaf Village Assistant so far answered in free text — fine for a chat window,
useless if another program needs to _use_ the answer. This notebook adds dynamic
`instructions`, tuned `ModelSettings`, and a structured `output_type`.


## Define the structured output shape

The SDK asks the model provider for JSON matching this schema, then validates and
parses it into a real instance for you.


In [ ]:
from dotenv import load_dotenv  # loads the .env file into the environment
from pydantic import (
    BaseModel,  # a Pydantic class that checks data has the right shape
)

load_dotenv()


# output_type= shape: passed to Agent(output_type=NinjaProfileCard) below, so
# final_output comes back as a checked object of this class and not as plain text
class NinjaProfileCard(BaseModel):
    """The exact shape the agent must return once output_type=NinjaProfileCard below."""

    name: str
    rank: str  # one of E, D, C, B, A, S
    team: str
    notable_skill: str


print("[schema description]", NinjaProfileCard.model_json_schema()["description"])

## Give the agent dynamic instructions

Instead of a fixed string, `instructions` can be a function
`(RunContextWrapper, Agent) -> str`, called fresh on every run — here it reads the
requester's own rank out of `context=`.


In [ ]:
from agents import (
    Agent,  # holds the instructions, tools, and model settings for one agent
    RunContextWrapper,  # holds the context= value you passed, and gives it to callbacks
)


# instructions function: passed to Agent(instructions=...) below. The SDK calls it
# on every run, so the persona can change based on context=
def build_ninja_profile_instructions(
    ctx: RunContextWrapper[dict[str, str]],  # holds this run's context= dict
    agent: Agent,  # the agent these instructions are for
) -> str:
    requester_rank = ctx.context.get("requester_rank", "unknown")
    return (
        f"You are the Leaf Village Assistant. A rank-{requester_rank} ninja is asking "
        "you to look up a ninja's profile. Return the full profile card — name, rank, "
        "team, and one notable skill."
    )

## Tune `ModelSettings` and assemble the agent

A profile lookup doesn't need deep reasoning or a long answer, so this agent uses
low effort and low verbosity.


In [3]:
from agents import (
    Agent,  # holds the instructions, tools, and model settings for one agent
    ModelSettings,  # settings for how the model behaves, like effort, timeout, and retries
    RunContextWrapper,  # holds the context= value you passed, and gives it to callbacks
)

# Initialize the agent
ninja_profile_agent = Agent(
    name="Leaf Village Assistant",
    # instructions: a function here instead of text. It runs again on every call
    instructions=build_ninja_profile_instructions,
    # model_settings: reasoning effort and verbosity, separate from the prompt text
    model_settings=ModelSettings(reasoning={"effort": "low"}, verbosity="low"),
    # output_type: final_output comes back as a checked NinjaProfileCard and not as text
    output_type=NinjaProfileCard,
)

print("[reasoning]", ninja_profile_agent.model_settings.reasoning)
print("[verbosity]", ninja_profile_agent.model_settings.verbosity)

# Call the instructions function directly (no run needed) to see what it gives back:
sample_ctx = RunContextWrapper(context={"requester_rank": "Kage"})
print("[instructions]", build_ninja_profile_instructions(sample_ctx, ninja_profile_agent))

[reasoning] Reasoning(context=None, effort='low', generate_summary=None, mode=None, summary=None)
[verbosity] low
[instructions] You are the Leaf Village Assistant. A rank-Kage ninja is asking you to look up a ninja's profile. Return the full profile card — name, rank, team, and one notable skill.


## Run it — offline and scripted

Script the model to reply with JSON matching `NinjaProfileCard`; `final_output`
should come back as an actual instance of it, not a string you'd have to
`json.loads` yourself.


In [ ]:
import json  # builds the scripted model's fixed JSON reply

from agents import (
    RunConfig,  # settings for one run, like model, workflow_name, and group_id
    Runner,  # runs an Agent: run, run_sync, or run_streamed
)
from agents.testing import (
    ScriptedModel,  # a fake model that gives set answers: no API key, no internet
    assistant_message,  # makes one scripted assistant reply for ScriptedModel
)


# Demo: scripts a JSON reply that fits NinjaProfileCard and proves final_output comes back as a real object
async def demo_structured_output() -> None:
    scripted_reply = json.dumps(
        {
            "name": "Naruto Uzumaki",
            "rank": "Kage",
            "team": "Team 7",
            "notable_skill": "Shadow Clone Summoning",
        }
    )
    # Script the model
    scripted_model = ScriptedModel(
        [
            [
                assistant_message(
                    scripted_reply,
                ),
            ]
        ]
    )

    # Run the Agent
    result = await Runner.run(
        ninja_profile_agent,
        "Look up Naruto Uzumaki's profile.",
        # context: the value that the instructions function reads with ctx.context
        context={"requester_rank": "Kage"},
        # run_config: settings for just this run
        run_config=RunConfig(model=scripted_model, workflow_name="e02-structured-output-demo"),
    )

    # Inspect the result
    print("[type]", type(result.final_output).__name__)
    print("[final]", result.final_output)
    # proves output_type= turns the model's JSON reply into a real NinjaProfileCard
    # object, so the caller does not have to run json.loads
    assert isinstance(result.final_output, NinjaProfileCard)


await demo_structured_output()
# Outside a notebook: asyncio.run(demo_structured_output())

## Run it live — optional, needs a real key


In [ ]:
import os  # reads OPENAI_API_KEY from the environment

from agents import (
    Runner,  # runs an Agent: run, run_sync, or run_streamed
)


# Demo: the same profile lookup with the real API, proving output_type= works on a real model reply and not only a scripted one
async def demo_live() -> None:
    if not os.environ.get("OPENAI_API_KEY"):
        print("[skipped] add a real OPENAI_API_KEY to .env to run this one for real")
        return

    # Run the Agent
    live_result = await Runner.run(
        ninja_profile_agent,
        "Look up Naruto Uzumaki's profile.",
        # context: your own value that tools and callbacks can read during the run
        context={"requester_rank": "Kage"},
    )
    print("[live]", live_result.final_output)


await demo_live()
# Outside a notebook: asyncio.run(demo_live())

## Summary

- `instructions` can be a function `(RunContextWrapper, Agent) -> str`, called fresh
  on every run — this is how an agent's persona adapts to who's asking.
- `Agent(output_type=SomeModel)` turns `final_output` from a string into a validated
  instance of `SomeModel`, with `ModelSettings` tuning how the model gets there.
